# 🛒 Retail Revenue Forecasting — Deep Learning
## LSTM · GRU · Bi-LSTM · Bi-GRU  |  Comparaison & Anti-Overfitting


## 1. Imports & Configuration

In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (
    LSTM, GRU, Bidirectional, Dense, Dropout,
    BatchNormalization, Input
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam

import warnings
warnings.filterwarnings('ignore')
tf.random.set_seed(42)
np.random.seed(42)

print("TensorFlow version:", tf.__version__)
print("GPU:", tf.config.list_physical_devices('GPU') or "CPU only")


## 2. Chargement & Prétraitement des données

In [ ]:

df = pd.read_csv('online_retail.csv', encoding='ISO-8859-1')
print("Shape brut:", df.shape)
df.head(3)


In [ ]:

# Nettoyage
df.dropna(inplace=True)
df = df[df['Quantity'] > 0]
df = df[df['UnitPrice'] > 0]
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['Revenue'] = df['Quantity'] * df['UnitPrice']
print("Shape propre:", df.shape)

# Agrégation journalière
daily = (df.groupby(df['InvoiceDate'].dt.date)['Revenue']
           .sum()
           .reset_index())
daily.columns = ['Date', 'Revenue']
daily['Date'] = pd.to_datetime(daily['Date'])
daily = daily.sort_values('Date').reset_index(drop=True)

print(f"Jours disponibles : {len(daily)}")
print(daily.describe())


In [ ]:

plt.figure(figsize=(14, 4))
plt.plot(daily['Date'], daily['Revenue'], color='steelblue', linewidth=1.2)
plt.title('Revenu journalier historique')
plt.xlabel('Date')
plt.ylabel('Revenu (£)')
plt.tight_layout()
plt.show()


## 3. Normalisation & Création des séquences

In [ ]:

WINDOW   = 30   # lookback (jours)
HORIZON  = 1    # prédiction (jours)
TEST_FRAC = 0.15

values = daily['Revenue'].values.reshape(-1, 1)
scaler = MinMaxScaler()
scaled = scaler.fit_transform(values)

def make_sequences(data, window, horizon=1):
    X, y = [], []
    for i in range(len(data) - window - horizon + 1):
        X.append(data[i : i + window])
        y.append(data[i + window : i + window + horizon])
    return np.array(X), np.array(y)

X, y = make_sequences(scaled, WINDOW, HORIZON)
y = y.reshape(-1, HORIZON)

split = int(len(X) * (1 - TEST_FRAC))
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

print(f"Train : X{X_train.shape}  y{y_train.shape}")
print(f"Test  : X{X_test.shape}  y{y_test.shape}")


## 4. Architectures des modèles

> **Anti-overfitting** : Dropout (0.3), BatchNormalization, kernel_regularizer L2, EarlyStopping, ReduceLROnPlateau.

In [ ]:

def build_lstm(input_shape, units=64, dropout=0.3, l2_reg=1e-4):
    model = Sequential([
        LSTM(units, return_sequences=True,
             kernel_regularizer=l2(l2_reg),
             input_shape=input_shape),
        BatchNormalization(),
        Dropout(dropout),
        LSTM(units // 2, kernel_regularizer=l2(l2_reg)),
        BatchNormalization(),
        Dropout(dropout),
        Dense(32, activation='relu'),
        Dropout(dropout / 2),
        Dense(1)
    ], name='LSTM')
    return model

def build_gru(input_shape, units=64, dropout=0.3, l2_reg=1e-4):
    model = Sequential([
        GRU(units, return_sequences=True,
            kernel_regularizer=l2(l2_reg),
            input_shape=input_shape),
        BatchNormalization(),
        Dropout(dropout),
        GRU(units // 2, kernel_regularizer=l2(l2_reg)),
        BatchNormalization(),
        Dropout(dropout),
        Dense(32, activation='relu'),
        Dropout(dropout / 2),
        Dense(1)
    ], name='GRU')
    return model

def build_bilstm(input_shape, units=64, dropout=0.3, l2_reg=1e-4):
    model = Sequential([
        Bidirectional(
            LSTM(units, return_sequences=True,
                 kernel_regularizer=l2(l2_reg)),
            input_shape=input_shape),
        BatchNormalization(),
        Dropout(dropout),
        Bidirectional(LSTM(units // 2,
                           kernel_regularizer=l2(l2_reg))),
        BatchNormalization(),
        Dropout(dropout),
        Dense(32, activation='relu'),
        Dropout(dropout / 2),
        Dense(1)
    ], name='Bi_LSTM')
    return model

def build_bigru(input_shape, units=64, dropout=0.3, l2_reg=1e-4):
    model = Sequential([
        Bidirectional(
            GRU(units, return_sequences=True,
                kernel_regularizer=l2(l2_reg)),
            input_shape=input_shape),
        BatchNormalization(),
        Dropout(dropout),
        Bidirectional(GRU(units // 2,
                          kernel_regularizer=l2(l2_reg))),
        BatchNormalization(),
        Dropout(dropout),
        Dense(32, activation='relu'),
        Dropout(dropout / 2),
        Dense(1)
    ], name='Bi_GRU')
    return model

INPUT_SHAPE = (WINDOW, 1)
print("Architectures définies ✓")


## 5. Entraînement des 4 modèles

In [ ]:

EPOCHS    = 100
BATCH     = 32
PATIENCE  = 12

def get_callbacks(name):
    return [
        EarlyStopping(monitor='val_loss', patience=PATIENCE,
                      restore_best_weights=True, verbose=0),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                          patience=6, min_lr=1e-6, verbose=0),
        ModelCheckpoint(f'{name}_best.keras', save_best_only=True,
                        monitor='val_loss', verbose=0)
    ]

def train_model(builder, name):
    model = builder(INPUT_SHAPE)
    model.compile(optimizer=Adam(learning_rate=1e-3), loss='mse',
                  metrics=['mae'])
    history = model.fit(
        X_train, y_train,
        validation_split=0.15,
        epochs=EPOCHS,
        batch_size=BATCH,
        callbacks=get_callbacks(name),
        verbose=0
    )
    print(f"  {name:10s} → {len(history.history['loss'])} époques "
          f"| val_mae={min(history.history['val_mae']):.4f}")
    return model, history

print("Entraînement en cours…")
lstm_model,  lstm_hist  = train_model(build_lstm,   'LSTM')
gru_model,   gru_hist   = train_model(build_gru,    'GRU')
bilstm_model,bilstm_hist= train_model(build_bilstm, 'BiLSTM')
bigru_model, bigru_hist = train_model(build_bigru,  'BiGRU')
print("✅ Tous les modèles entraînés !")


## 6. Courbes d'apprentissage (détection overfitting)

In [ ]:

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
configs = [
    (lstm_hist,   'LSTM',    axes[0,0]),
    (gru_hist,    'GRU',     axes[0,1]),
    (bilstm_hist, 'Bi-LSTM', axes[1,0]),
    (bigru_hist,  'Bi-GRU',  axes[1,1]),
]
for hist, name, ax in configs:
    ax.plot(hist.history['loss'],     label='Train Loss', linewidth=2)
    ax.plot(hist.history['val_loss'], label='Val Loss',   linewidth=2, linestyle='--')
    ax.set_title(f'{name} — Loss (MSE)', fontsize=13, fontweight='bold')
    ax.set_xlabel('Époque')
    ax.set_ylabel('MSE')
    ax.legend()
    ax.grid(alpha=0.3)

fig.suptitle('Courbes de perte — Anti-overfitting avec Dropout + BatchNorm + L2',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


## 7. Évaluation sur le jeu de test

In [ ]:

def evaluate(model, name):
    pred_scaled = model.predict(X_test, verbose=0)
    pred = scaler.inverse_transform(pred_scaled)
    actual = scaler.inverse_transform(y_test)
    mae  = mean_absolute_error(actual, pred)
    rmse = np.sqrt(mean_squared_error(actual, pred))
    mape = np.mean(np.abs((actual - pred) / (actual + 1e-8))) * 100
    return {'Modèle': name, 'MAE': round(mae,2),
            'RMSE': round(rmse,2), 'MAPE (%)': round(mape,2),
            '_pred': pred, '_actual': actual}

results = [
    evaluate(lstm_model,   'LSTM'),
    evaluate(gru_model,    'GRU'),
    evaluate(bilstm_model, 'Bi-LSTM'),
    evaluate(bigru_model,  'Bi-GRU'),
]

metrics_df = pd.DataFrame([{k:v for k,v in r.items() if not k.startswith('_')}
                            for r in results])
metrics_df = metrics_df.sort_values('RMSE')
print(metrics_df.to_string(index=False))


## 8. Comparaison visuelle des métriques

In [ ]:

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']
models = metrics_df['Modèle'].tolist()

for ax, metric in zip(axes, ['MAE', 'RMSE', 'MAPE (%)']):
    vals = metrics_df[metric].tolist()
    bars = ax.bar(models, vals, color=colors, edgecolor='white', linewidth=0.8)
    ax.set_title(metric, fontsize=13, fontweight='bold')
    ax.set_ylabel(metric)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(vals)*0.01,
                f'{val:.2f}', ha='center', va='bottom', fontsize=10)
    ax.grid(axis='y', alpha=0.3)

fig.suptitle('📊 Comparaison LSTM · GRU · Bi-LSTM · Bi-GRU',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


## 9. Prédictions vs Valeurs réelles

In [ ]:

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes = axes.flatten()

for i, (res, ax) in enumerate(zip(results, axes)):
    actual = res['_actual'].flatten()
    pred   = res['_pred'].flatten()
    ax.plot(actual, label='Réel',       color='#2196F3', linewidth=1.5)
    ax.plot(pred,   label='Prédit',     color='#FF5722', linewidth=1.5, linestyle='--')
    ax.fill_between(range(len(actual)), actual, pred,
                    alpha=0.15, color='orange', label='Écart')
    ax.set_title(f"{res['Modèle']}  |  MAE={res['MAE']:,.0f}  RMSE={res['RMSE']:,.0f}",
                 fontsize=12, fontweight='bold')
    ax.set_xlabel('Jours (test)')
    ax.set_ylabel('Revenu (£)')
    ax.legend()
    ax.grid(alpha=0.3)

fig.suptitle('Prédictions vs Valeurs réelles — Jeu de test',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


## 10. Radar Chart — synthèse des performances

In [ ]:

from matplotlib.patches import FancyBboxPatch

# Normaliser les métriques (plus bas = meilleur → inverser)
norm = metrics_df[['MAE','RMSE','MAPE (%)']].copy()
for col in norm.columns:
    norm[col] = 1 - (norm[col] - norm[col].min()) / (norm[col].max() - norm[col].min() + 1e-9)

categories = ['MAE Score', 'RMSE Score', 'MAPE Score']
N = len(categories)
angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
colors_r = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']

for i, (_, row) in enumerate(metrics_df.iterrows()):
    name = row['Modèle']
    scores = norm.loc[_].tolist() if isinstance(_, int) else norm.iloc[i].tolist()
    scores += scores[:1]
    ax.plot(angles, scores, 'o-', linewidth=2, label=name, color=colors_r[i])
    ax.fill(angles, scores, alpha=0.1, color=colors_r[i])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=12)
ax.set_ylim(0, 1)
ax.set_title('Radar — Performance relative (1 = meilleur)',
             fontsize=13, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
plt.tight_layout()
plt.show()


## 11. Prédiction des 30 prochains jours

In [ ]:

FUTURE_DAYS = 30

def forecast_future(model, last_window, n_days):
    preds = []
    window = last_window.copy()
    for _ in range(n_days):
        p = model.predict(window.reshape(1, WINDOW, 1), verbose=0)[0, 0]
        preds.append(p)
        window = np.append(window[1:], [[p]], axis=0)
    return scaler.inverse_transform(np.array(preds).reshape(-1, 1)).flatten()

last_window = scaled[-WINDOW:]

future_dates = pd.date_range(
    start=daily['Date'].iloc[-1] + pd.Timedelta(days=1),
    periods=FUTURE_DAYS
)

forecasts = {
    'LSTM':    forecast_future(lstm_model,   last_window, FUTURE_DAYS),
    'GRU':     forecast_future(gru_model,    last_window, FUTURE_DAYS),
    'Bi-LSTM': forecast_future(bilstm_model, last_window, FUTURE_DAYS),
    'Bi-GRU':  forecast_future(bigru_model,  last_window, FUTURE_DAYS),
}

fig, ax = plt.subplots(figsize=(14, 5))
hist_tail = daily.tail(60)
ax.plot(hist_tail['Date'], hist_tail['Revenue'],
        color='gray', linewidth=2, label='Historique', alpha=0.7)

for (name, vals), color in zip(forecasts.items(), colors_r):
    ax.plot(future_dates, vals, linewidth=2, label=name, color=color)

ax.axvline(daily['Date'].iloc[-1], color='black', linestyle=':', linewidth=1.5,
           label='Fin historique')
ax.fill_betweenx([0, ax.get_ylim()[1] if ax.get_ylim()[1] > 0 else 60000],
                 future_dates[0], future_dates[-1], alpha=0.05, color='green')
ax.set_title('🔮 Prédiction — 30 prochains jours (tous modèles)',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Revenu prédit (£)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

future_df = pd.DataFrame({'Date': future_dates, **forecasts})
print(future_df.to_string(index=False))


## 12. Sauvegarde du meilleur modèle

In [ ]:

best_name = metrics_df.iloc[0]['Modèle']
best_model_map = {
    'LSTM':    lstm_model,
    'GRU':     gru_model,
    'Bi-LSTM': bilstm_model,
    'Bi-GRU':  bigru_model,
}
best_model = best_model_map[best_name]
best_model.save('retail_forecasting_model.h5')
print(f"✅ Meilleur modèle sauvegardé : {best_name} → retail_forecasting_model.h5")
print(f"   MAE={metrics_df.iloc[0]['MAE']}  RMSE={metrics_df.iloc[0]['RMSE']}  MAPE={metrics_df.iloc[0]['MAPE (%)']}%")


## ✅ Résumé

| Modèle | Description | Points forts |
|--------|-------------|--------------|
| **LSTM** | Long Short-Term Memory | Bonne mémoire longue distance |
| **GRU** | Gated Recurrent Unit | Plus léger, entraînement plus rapide |
| **Bi-LSTM** | LSTM Bidirectionnel | Contexte passé + futur de la séquence |
| **Bi-GRU** | GRU Bidirectionnel | Meilleur équilibre performance/vitesse |

### Anti-overfitting mis en place
- **Dropout (30%)** sur toutes les couches récurrentes et denses
- **BatchNormalization** après chaque couche récurrente
- **L2 Regularization** sur les poids des kernels
- **EarlyStopping** (patience=12) avec restauration des meilleurs poids
- **ReduceLROnPlateau** (facteur 0.5, patience=6)
